# Reconstrucción v2
Nueva ejecución de la receta auditada; no se reutilizan salidas ni run_id de v1. Misma entrada congelada y folds. La selección de modelos incluirá seis candidatos y el criterio del negocio desde el diseño. 2014J es un benchmark previamente observado.

# 01 · EDA de desarrollo: decisiones antes de modelar

Wilmer Puerta · PSet 3 · Día 56

Se analizan **solo las matrículas de desarrollo congeladas**. La prueba 2014J permanece reservada. Las tablas y figuras son descriptivas: no se ajustan modelos ni imputadores ni se calcula rendimiento en hold-out. Las 37 columnas cargadas son las 35 originales OBT más `target_class` y `cv_fold`; el fold, las claves y los campos de auditoría no son predictores.

La preparación de modelos permanece centralizada en `src/pipeline.py`. Este notebook importa esa preparación y la carga verificada; el cálculo estadístico reproducible vive en `src/eda.py`.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, subprocess
BASE = Path.cwd()
if BASE.name == 'notebooks':
    BASE = BASE.parent
if not (BASE / 'src').is_dir():
    BASE = BASE / 'pset-03'
assert (BASE / 'src').is_dir(), 'Abrir desde notebooks, pset-03 o raíz del repositorio'
sys.path.insert(0, str(BASE))
os.environ['MPLCONFIGDIR'] = str(BASE / '.local/matplotlib')
from IPython.display import display, Markdown
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from src.datos import cargar_snapshot, pliegues_congelados
from src.pipeline import construir_pipeline
from src.eda import analizar, figuras, exportar, hallazgos
from src.common import read_bundle, sha256
from src.tracking import configurar
pd.set_option('display.max_rows', 45)
pd.set_option('display.max_columns', 16)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 1. Población y procedencia

Una fila por matrícula en día 56. Se preservan los alumnos con varias matrículas y se informa también la cantidad de personas. No se muestran filas individuales ni identificadores. La selección corresponde a la cohorte común elegible al día 56 y no se extrapola a quienes ya se retiraron antes de ese corte.

In [ ]:
train, snapshot_id = cargar_snapshot()
assert train.attrs['partition'] == 'train'
cv = list(pliegues_congelados(train))
result = analizar(train)
s = result['summary']
t = result['tables']
display(pd.DataFrame([{'matriculas': s['rows'], 'alumnos_unicos': s['unique_students'],
                       'columnas_OBT': s['original_obt_columns'], 'columnas_cargadas': s['columns_loaded'],
                       'corte_dia': s['cutoff_day'], 'folds_congelados': len(cv)}]))
print('snapshot_id:', snapshot_id)
print('split_id:', s['split_id'])
print('Instante de extracción UTC:', s['snapshot_at_utc'])
display(pd.DataFrame([{'control': k, 'aprobado': v} for k,v in s['checks'].items()]))

## 2. Tipos, roles, faltantes y descriptivos

El perfil cubre **todas las columnas cargadas**. Los descriptivos numéricos se limitan a variables candidatas permitidas por el contrato: no se correlacionan IDs ni códigos ni números de fold. Se conservan los tipos originales y los faltantes. Las frecuencias categóricas no convierten sus categorías en números ordinales arbitrarios.

In [ ]:
display(t['perfil_columnas'])
display(t['descriptivos_numericos'].round(3))
display(t['categorias'])

In [ ]:
plots = figuras(train, result)
display(plots['02_faltantes'])
plt.close(plots['02_faltantes'])
display(t['faltantes_target'].round(4))

`recency_days` y `last_activity_day` faltan en las mismas matrículas sin registros. El último es auditoría; no se introducen ambos como predictores equivalentes. La razón actual/anterior es indefinida con denominador cero. IMD conserva su ausencia. No se imputan estas columnas durante EDA: la política se ajustará dentro de cada fold.

Una recencia mayor que 56 puede corresponder a la última interacción **antes del inicio**; no se convierte automáticamente en error. Cero clics registrados tampoco demuestra cero actividad académica real.

## 3. Target y desbalance

El target multiclase distingue `Approved` (Pass + Distinction), `Fail` y `Withdrawn`. La métrica principal futura se centra en P(Fail). No se transforma el target multiclase en una escala ordinal.

Las referencias de prevalencia y clase mayoritaria se calculan de forma descriptiva: **no son todavía corridas de baseline entrenadas**. El desbalance motiva métricas por clase y AP para Fail; no justifica por sí solo sobremuestreo ni SMOTE global.

In [ ]:
display(t['target'].round(3))
display(plots['01_target']); plt.close(plots['01_target'])
print(f"Razón clase mayoritaria/minoritaria: {s['majority_minority_ratio']:.3f}")
print(f"Referencia accuracy de clase mayoritaria: {100*s['majority_class_accuracy_reference']:.2f} %")
print(f"Prevalencia Fail (referencia para interpretar AP): {100*s['fail_prevalence']:.2f} %")
display(t['balance_folds'].round(4))

## 4. Distribuciones y valores atípicos

IQR: marcar observaciones estrictamente fuera de [Q1−1,5×IQR, Q3+1,5×IQR]. Z-score: |(x−media)/σ| **≥3**, con σ poblacional (`ddof=0`) y solo valores observados. Con varianza cero, Z no está definido. Se informa también IQR cero: en variables discretas concentradas en cero puede marcar todos los valores positivos legítimos.

La tabla usa los cuartiles de todo desarrollo; los boxplots recalculan cuartiles dentro de cada clase, por lo que sus puntos extremos no son los conteos globales de la tabla. Los histogramas usan los mismos bordes de bins para las tres clases en cada variable.

Estas reglas describen extremos; no prueban error de datos ni normalidad. No se elimina ninguna matrícula. Las transformaciones de los histogramas son únicamente visuales; los boxplots mantienen los valores originales y usan escala symlog cuando facilita ver cajas y colas sin ocultar extremos.

In [ ]:
display(t['outliers'].round(3))
display(plots['03_distribuciones']); plt.close(plots['03_distribuciones'])
display(plots['04_boxplots_iqr']); plt.close(plots['04_boxplots_iqr'])
print('Filas eliminadas por outliers:', s['outlier_rows_removed'])

## 5. Relación entre características y Fail

Se usa **Spearman** porque varios conteos y razones son asimétricos, contienen extremos y no sugieren una relación lineal. Se correlacionan los rangos con `Fail`=1 frente al resto=0; los empates se resuelven por rangos medios. Cada coeficiente usa observados por pareja y se informa su N. Una variable con varianza cero no recibe un coeficiente artificial.

La matriz completa se exporta. La figura muestra diez asociaciones marginales de mayor magnitud para legibilidad: **esa visualización no selecciona automáticamente features**. No se calculan p-valores ni se afirma causalidad; varias matrículas pertenecen a un mismo alumno y la composición por módulo puede confundir estas relaciones. Las categóricas se examinan por frecuencias y tasas, sin codificación ordinal arbitraria.

In [ ]:
display(t['correlacion_fail'].round(4))
display(plots['05_correlaciones_spearman']); plt.close(plots['05_correlaciones_spearman'])
display(t['medianas_target'].round(3))
display(t['redundancia'].round(4))

Correlaciones altas entre acumulados y ventanas sugieren redundancia, pero no bastan para borrar variables. Se comparará el aporte con CV congelada y ablaciones. La mediana del cambio puede coincidir entre clases aunque sus distribuciones difieran; una única medida central no describe todo el patrón.

## 6. Registros de actividad y contexto

Las tasas siguientes describen desarrollo y no son métricas de un clasificador. No tener registros no equivale a no estudiar. Se distingue ausencia hasta el corte de ausencia únicamente en la ventana reciente. Los códigos de módulo y presentación se usan para contextualizar heterogeneidad y evaluar posteriormente; no se incorporan automáticamente como predictores.

In [ ]:
display(t['actividad_registrada'].round(4))
display(t['estado_actividad'].round(4))
display(plots['07_estado_actividad']); plt.close(plots['07_estado_actividad'])
display(t['tasas_modulo'].round(4)); display(t['tasas_presentacion'].round(4))
display(plots['06_contexto_modulo_presentacion']); plt.close(plots['06_contexto_modulo_presentacion'])

## 7. Hallazgos y decisiones para la etapa de features

La tabla enlaza evidencia con acciones. Los prototipos `log1p_clicks_current_14d` y `clicks_per_active_day_14d` usan solo registros disponibles hasta el corte; la intensidad deja NULL si no hay días activos en la ventana. Se muestran sus medianas observadas, **sin afirmar que aporten capacidad predictiva**. La próxima etapa implementará las transformaciones y probará su aporte dentro de CV; no se modificó todavía `src/pipeline.py`.

In [ ]:
findings = hallazgos(result)
rows = ['| Hallazgo | Evidencia | Consecuencia en el experimento |', '|---|---|---|']
for finding in findings:
    rows.append('| ' + ' | '.join(str(finding[k]).replace('|', '\|') for k in ['hallazgo','evidencia','consecuencia']) + ' |')
display(Markdown(chr(10).join(rows)))
display(t['candidatas_derivadas'].round(4))

## 8. Exportación y trazabilidad en MLflow

Se guardan tablas y figuras agregadas en `docs/eda/`, junto con IDs de snapshot/split y hashes. No se publican filas individuales. Se registra una corrida de fase `eda`, adicional al presupuesto mínimo de modelos: no contiene un modelo entrenado y no se contabiliza como baseline/candidato/tuning/final. Cada Restart/Run All genera una corrida EDA distinta con la misma identidad de datos y fuentes; la fecha y el run_id distinguen ejecuciones.

In [ ]:
output = exportar(result, plots)
import mlflow
experiment = configurar()
commit = subprocess.run(['git','rev-parse','--verify','HEAD'], cwd=BASE.parent, capture_output=True, text=True)
commit_id = commit.stdout.strip() if commit.returncode == 0 else 'uncommitted'
dirty = False # registrar_base exige fuente/config v2 limpias en Git antes del run
source_nb = __import__('nbformat').read(BASE/'notebooks/01_eda.ipynb', as_version=4)
notebook_source_hash = hashlib.sha256(json.dumps([(c.cell_type,c.source) for c in source_nb.cells],ensure_ascii=False).encode()).hexdigest()
from src.seguimiento import registrar_base
with mlflow.start_run(run_name='eda-desarrollo-dia56') as run:
    registrar_base('eda', 'analisis_descriptivo', require_frozen=False)
    mlflow.set_tags({'fase':'eda','familia':'analisis_descriptivo','cuenta_para_presupuesto_minimo':'false',
                     'git_commit':commit_id, 'git_dirty':str(dirty).lower(), 'partition':'train',
                     'eda_source_sha256':sha256(BASE/'src/eda.py'),'notebook_source_sha256':notebook_source_hash})
    mlflow.log_params({'snapshot_id':s['snapshot_id'],'split_id':s['split_id'],'snapshot_at_utc':s['snapshot_at_utc'],
                       'source_table':'PSET2_OULAD.ANALYTICS.OBT_FEATURES','labels_table':'PSET2_OULAD.GOLD.OUTCOME_ENROLLMENT',
                       'cutoff_day':56,'seed':42,'correlation':'spearman','zscore_ddof':0,'outlier_iqr_multiplier':1.5,
                       'client_lock_sha256':sha256(BASE/'requirements.lock.txt')})
    mlflow.log_metrics({'development_rows':s['rows'],'development_unique_students':s['unique_students'],
                        'train_fail_prevalence':s['fail_prevalence'],'outlier_rows_removed':0})
    s.update(run_id=run.info.run_id, experiment_id=experiment.experiment_id,
             notebook_source_sha256=notebook_source_hash, git_commit=commit_id, git_dirty=dirty)
    (output/'resumen.json').write_text(json.dumps(s,ensure_ascii=False,indent=2,allow_nan=False)+chr(10))
    mlflow.log_artifacts(str(output),artifact_path='eda')
    mlflow.log_artifact(str(BASE/'src/eda.py'),artifact_path='source')
    mlflow.log_artifact(str(BASE/'notebooks/01_eda.ipynb'),artifact_path='source')
assert mlflow.MlflowClient().get_run(s['run_id']).info.status == 'FINISHED'
print('Corrida EDA FINISHED:', s['run_id'])
print('Tablas y figuras agregadas:', output.relative_to(BASE))
print('Esta ejecución v2 solo analiza desarrollo; 2014J fue observado en v1 y se declara como benchmark reutilizado.')

## Cierre y límites

EDA completada exclusivamente en desarrollo. Las asociaciones se usan para proponer experimentos y no son resultados de generalización. Las etiquetas de OULAD no incluyen timestamps de publicación; los registros no prueban cobertura completa de actividad real. La comparación prospectiva institucional y el efecto de tutorías siguen fuera de esta evidencia.

**Siguiente paso:** dos features derivadas, disponibilidad al inferir y ablación con folds congelados; política de faltantes/codificación dentro de `Pipeline` y `ColumnTransformer`. El hold-out seguirá reservado hasta fijar los dos finalistas.